In [ ]:
# Step 1 - Load environment variables and API key


import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if api_key:
    print("API key loaded successfully!")
    print(f"Key starts with: {api_key[:8]}...")
else:
    print("API key missing - check your .env file")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=api_key
)


response = llm.invoke("Say hello and introduce yourself briefly")
print("\nLLM Response:")
print(response.content)

API key loaded successfully!
Key starts with: gsk_c4wF...

LLM Response:
Hello! I'm an AI assistant, and I'm here to help answer your questions and provide information on a wide range of topics. I don't have a personal name, but I'm often referred to as a conversational AI or a chatbot. I'm looking forward to chatting with you and helping with anything you need. How can I assist you today?


In [4]:
# Step 2 - Load PDF documents from data folder

from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("../data/")
documents = loader.load()

print(f"Total pages loaded: {len(documents)}")
print(f"\nFirst document snippet:")
print(documents[0].page_content[:300])

Total pages loaded: 237

First document snippet:
Maneka Gandhi vs Union Of India on 25 January, 1978
Equivalent citations: 1978 AIR 597, 1978 SCR (2) 621
Author: M. Hameedullah Beg
Bench: M. Hameedullah Beg, Y.V. Chandrachud, P.N. Bhagwati, V.R.
Krishnaiyer, N.L. Untwalia, Syed Murtaza Fazalali, P.S. Kailasam
           PETITIONER:
MANEKA GANDHI
 


In [5]:
# Step 3 - Split documents into 

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\nExample chunk:")
print(chunks[0].page_content)
print(f"\nChunk metadata:")
print(chunks[0].metadata)

Total chunks created: 1064

Example chunk:
Maneka Gandhi vs Union Of India on 25 January, 1978
Equivalent citations: 1978 AIR 597, 1978 SCR (2) 621
Author: M. Hameedullah Beg
Bench: M. Hameedullah Beg, Y.V. Chandrachud, P.N. Bhagwati, V.R.
Krishnaiyer, N.L. Untwalia, Syed Murtaza Fazalali, P.S. Kailasam
           PETITIONER:
MANEKA GANDHI
        Vs.
RESPONDENT:
UNION OF INDIA
DATE OF JUDGMENT25/01/1978
BENCH:
BEG, M. HAMEEDULLAH (CJ)
BENCH:
BEG, M. HAMEEDULLAH (CJ)
CHANDRACHUD, Y.V.
BHAGWATI, P.N.
KRISHNAIYER, V.R.
UNTWALIA, N.L.
FAZALALI, SYED MURTAZA
KAILASAM, P.S.
CITATION:
 1978 AIR  597            1978 SCR  (2) 621
 1978 SCC  (1) 248
 CITATOR INFO :
 R          1978 SC1514  (12)
 E&R        1978 SC1548  (4,10,23)
 R          1978 SC1594  (6,15)
 E&R        1978 SC1675  (53,55,57,127,167,171,197,227,
 E&R        1979 SC 478  (90,91A,129,159)
 D          1979 SC 745  (20,30,36,37,52,77)
 RF         1979 SC 916  (15,54)
 R          1979 SC1360  (2,5)
 R          1979 SC1369  (6)
 R

In [6]:
# Step 4 - Create embeddings and store in ChromaDB

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Creating vector store and embedding all chunks...")
print("This will take a few minutes, please wait...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../chroma_db"
)

print(f"\nVector store created successfully!")
print(f"Total vectors stored: {vectorstore._collection.count()}")

Loading embedding model...


e:\Random-Projects\Legal Assist Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3094.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating vector store and embedding all chunks...
This will take a few minutes, please wait...

Vector store created successfully!
Total vectors stored: 1064


In [7]:
# Step 5 - Create the retriever

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

test_query = "What are the fundamental rights of a person?"
retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} chunks\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Content: {doc.page_content[:200]}")
    print()

Retrieved 5 chunks

--- Chunk 1 ---
Source: ..\data\Maneka_Gandhi_vs_Union_Of_India_on_25_January_1978.PDF
Content: and educational rights , (vi) right to property, and (vii) right to constitutional
remedies. They are the rights of the people preserved by our Constitution,
'Fundamental rights' are the modem name fo

--- Chunk 2 ---
Source: ..\data\Maneka_Gandhi_vs_Union_Of_India_on_25_January_1978.PDF
Content: which are really impediments to the exercise of the "inalienable" rights' Such deprivations or
restrictions or regulations of rights may take place, within prescribed limits, by means of either
statut

--- Chunk 3 ---
Source: ..\data\Maneka_Gandhi_vs_Union_Of_India_on_25_January_1978.PDF
Content: right to go abroad. It cannot be disputed that there must exist a basically free sphere for man,
resulting from the nature and dignity of the human being as the bearer of the highest spiritual and
mor

--- Chunk 4 ---
Source: ..\data\Maneka_Gandhi_vs_Union_Of_India_on_25_January_1978.PDF

In [8]:
# Step 6 - Test with a question

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Provide the prompt format
template = """
You are an expert legal assistant specializing in Indian law and Supreme Court judgments. 
Use the following context from actual court documents to answer the question accurately.
If the answer is not in the context, say "I don't have enough information in my documents to answer this accurately."

Context:
{context}

Question: {question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

# 2. Add a helper function to merge chunks of text together
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Create the modern LCEL RAG chain
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Test query
query = "What are the fundamental rights of a person under Indian law?"
print("QUESTION:")
print(query)

# Notice we use .invoke() directly with the string now!
result = qa_chain.invoke(query)

print("\nANSWER:")
print(result)


QUESTION:
What are the fundamental rights of a person under Indian law?

ANSWER:
According to the context, the fundamental rights of a person under Indian law are embodied in Part III of the Constitution and can be classified as follows: 

1. Right to equality
2. Right to freedom
3. Right against exploitation
4. Right to freedom of religion
5. Cultural and educational rights

Additionally, Articles 19 to 22 provide for different aspects of freedom, including:

- Right to freedom of speech and expression (Article 19(1)(a))
- Right to assemble peaceably and without arms (Article 19(1)(b))
- Right to form associations or unions (Article 19(1)(c))
- Right to move freely throughout the territory of India (Article 19(1)(d))
- Right to reside and settle in any part of the territory of India (Article 19(1)(e))
- Right to acquire, hold and dispose of property (Article 19(1)(f))
- Right to practise any profession or to carry on any occupation, trade or business (Article 19(1)(g))

These rights a